# P4 Tourism Monthly — bounded Round 2 discovery

只上传这个 notebook 即可；无需另外上传数据 ZIP、同步仓库或设置 GH_TOKEN。
Colab 选择 **T4 GPU**，默认 `BACKBONE = 'both'`、`MODE = 'full'`，然后全部运行。
每个 backbone 会先验证/补跑 smoke，再运行 full；两者完成后自动分析四个 cells。
想先测速度，可把 MODE 改成 'smoke'。只重做汇总用 'analyze'，无需 GPU。
断线后保持设置和输出目录，重新全部运行；已校验的单位会恢复，不删旧结果。

This notebook embeds code/config only. It downloads the official Tourism Monthly
archive (about 200 kB), verifies its SHA-256, and replays the existing 32-ID discovery
assignment and 192 construct checks. The archive contains reserved B observations,
but B future values are never numerically parsed, sent to models, or used for scoring.
No dataset values or per-ID selection receipts are embedded in this notebook.

**Scope:** 32 original A IDs, three seeds, two mechanisms, two frozen backbones.
The 96 responses per cell are 32 ID clusters, not 96 independent original series.
All four cells and unsuccessful screens remain reportable. The unchanged P1 thresholds
are exploratory screens here, not an independent-confirmation claim. Cross-series
dependence/pretraining overlap remains unresolved. Native monthly cadence means
192 months of context and a 24-month forecast horizon. The oracle describes the
synthetically added response, not a native causal tourism effect.

Official source: https://zenodo.org/records/4656096 (CC BY 4.0).
TimesFM-3 retains its non-commercial licence. HF_TOKEN is optional for public downloads;
if configured, enable notebook access in Colab Secrets. Never paste tokens into cells.

Outputs: `MyDrive/tsfm-covariate-faithfulness/p4_tourism_discovery_auto_v1/`.
This notebook does not change the electricity archive or the manuscript.


In [ ]:
BACKBONE = 'both'  # both (recommended), chronos_2, or timesfm_3
MODE = 'full'  # full automatically runs/checks smoke; smoke or analyze also available
assert BACKBONE in {'both', 'chronos_2', 'timesfm_3'}
assert MODE in {'smoke', 'full', 'analyze'}
print('P4 tourism discovery:', BACKBONE, MODE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile

# Code/config only, not tourism observations or ID-selection receipts.
EXPECTED_SOURCE_SHA256 = '0136fa0192db1b55dfe92510642445f2dc88fc3a22e7780eb2547dc14bf43883'
CODE_ZIP_BASE64 = '...'
payload = base64.b64decode(CODE_ZIP_BASE64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SOURCE_SHA256, 'Code snapshot checksum failed'
BOOT = Path('/content/p4_tourism_bootstrap_' + EXPECTED_SOURCE_SHA256[:12])
BOOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for entry in archive.infolist():
        target = (BOOT / entry.filename).resolve()
        if not target.is_relative_to(BOOT.resolve()):
            raise RuntimeError('Unsafe source snapshot path')
        content = archive.read(entry)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError('Local bootstrap changed. Use a fresh runtime; do not bypass checks.')
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(content)
    expected_py = {n for n in archive.namelist() if n.endswith('.py')}
    actual_py = {p.relative_to(BOOT).as_posix() for p in (BOOT / 'src').rglob('*.py')}
    assert actual_py == expected_py, 'Unexpected stale Python module in bootstrap'
PROJECT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness')
CACHE = PROJECT / 'p4_official_source_cache'
ROOT = PROJECT / ('p4_tourism_runtime_' + EXPECTED_SOURCE_SHA256[:12])
OUTPUT = PROJECT / 'p4_tourism_discovery_auto_v1'
print('Code snapshot verified:', EXPECTED_SOURCE_SHA256)
print('Source cache:', CACHE)
print('Durable experiment output:', OUTPUT)

def run_logged(command, *, cwd, env, log_name):
    log_path = OUTPUT / 'logs' / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Stopped with exit code {returncode}. Copy the actual error above. Log: {log_path}')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(BOOT / 'requirements/colab-base.txt')], check=True)
if MODE != 'analyze':
    gpu_check = ('import torch; '
                 'assert torch.cuda.is_available(), "Select T4 GPU and reconnect"; '
                 'print(torch.cuda.get_device_name(0), torch.__version__)')
    subprocess.run([sys.executable, '-c', gpu_check], check=True)
bootstrap_env = os.environ.copy()
bootstrap_env['PYTHONPATH'] = str(BOOT / 'src')
run_logged([sys.executable, '-u', '-m', 'covfaith_p4.tourism_autodata',
            '--bootstrap', str(BOOT), '--cache', str(CACHE), '--runtime', str(ROOT)],
           cwd=BOOT, env=bootstrap_env, log_name='automatic_data_setup.log')
manifest = json.loads((ROOT / 'release_manifest.json').read_text())
print('Tourism data/selection/constructs verified. Config:', manifest['config_hash'])


In [ ]:
if MODE != 'analyze':
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print('HF_TOKEN unavailable; public-model access is still allowed.')
    else:
        os.environ['HF_TOKEN'] = hf_token
        del hf_token
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(ROOT / 'src')
import_check = ('from pathlib import Path; '
                'from covfaith_p4.tourism_pilot import load_release; '
                'load_release(Path.cwd()); print("Tourism imports and release checks OK")')
subprocess.run([sys.executable, '-c', import_check], cwd=ROOT, env=ENV, check=True)


In [ ]:
# Separate subprocesses release GPU memory and avoid stale imported model packages.
backbones = ['chronos_2', 'timesfm_3'] if BACKBONE == 'both' else [BACKBONE]
if MODE != 'analyze':
    for backbone in backbones:
        if backbone == 'timesfm_3':
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=True)
        req = 'chronos2.txt' if backbone == 'chronos_2' else 'timesfm3.txt'
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                        str(ROOT / 'requirements' / req)], check=True)
        if backbone == 'timesfm_3':
            check = ('from timesfm3 import ModelConfig, TimesFM3Evaluator; '
                     'print("TimesFM-3 import OK")')
            subprocess.run([sys.executable, '-c', check], check=True)
        gpu_check = ('import torch; assert torch.cuda.is_available(), '
                     '"GPU unavailable; select T4 and reconnect"')
        subprocess.run([sys.executable, '-c', gpu_check], check=True)
        run_logged([sys.executable, '-u', '-m', 'covfaith_p4.tourism_pilot',
                    '--root', str(ROOT), '--output', str(OUTPUT), '--mode', MODE,
                    '--backbone', backbone], cwd=ROOT, env=ENV, log_name=f'{backbone}_{MODE}.log')
ANALYSIS_RAN = MODE == 'analyze' or (MODE == 'full' and BACKBONE == 'both')
if ANALYSIS_RAN:
    run_logged([sys.executable, '-u', '-m', 'covfaith_p4.tourism_pilot',
                '--root', str(ROOT), '--output', str(OUTPUT), '--mode', 'analyze'],
               cwd=ROOT, env=ENV, log_name='analysis.log')


In [ ]:
if ANALYSIS_RAN:
    report_path = OUTPUT / 'analysis/discovery_report.json'
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2))
    print('Send discovery_report.json and complete_four_cell_matrix.csv, not private NPZ arrays.')
    from google.colab import files
    EXPORT = PROJECT / 'p4_tourism_discovery_summary_v1.zip'
    with zipfile.ZipFile(EXPORT, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for name in ('discovery_report.json', 'complete_four_cell_matrix.csv'):
            bundle.write(OUTPUT / 'analysis' / name, arcname=name)
    files.download(str(EXPORT))
    print('Summary ZIP also saved on Drive:', EXPORT)
else:
    for backbone in backbones:
        report_path = OUTPUT / MODE / backbone / 'completion.json'
        report = json.loads(report_path.read_text())
        print(json.dumps(report, indent=2))
        if MODE == 'smoke':
            estimate = report['linear_full_time_projection_seconds_not_guarantee'] / 60
            print(f'{backbone}: rough full-run projection {estimate:.1f} minutes; not a guarantee.')
    print('After smoke, use MODE=full and BACKBONE=both to run/resume both and analyze.')
print('Private output directory:', OUTPUT)
